# 4.0 本章表格的讀法

全章各表共用同一組欄位。先一次講清楚，後續不再重複。

| 欄位 | 意義 | 如何得到 |
| :--- | :--- | :--- |
| **pp**（百分點） | **兩個百分比相減**的單位。年化 1.5% 對 0.7% 的差是 **0.8 pp**，不是 0.8% | 差值本身 |
| **年化Δ** | 處理組減對照組的**年化報酬差**，正值代表處理組較優 | 逐日差分 $\Delta r_t$ 的平均 × 252 |
| **95% CI** | 效果量的信賴區間 | bootstrap 分布的 2.5／97.5 百分位（§3.5） |
| **$p$** | 雙尾 $p$ 值 | 將 bootstrap 分布平移至中心為零後，觀測值的極端程度 |
| **BH 校正 $p$** | 多重比較校正後的 $p$ | Benjamini-Hochberg 控制 FDR；同一項檢定的各組一起校正 |
| **IR**（資訊比率） | 差分序列的**風險調整後**效果 | 年化Δ ÷ 差分序列的年化標準差 |
| **勝日%** | 處理組當日報酬高於對照組的交易日占比 | $\#\{\Delta r_t > 0\}$ ÷ 交易日數 |
| **逐格顯著** | 15 個參數格中，**單獨**檢定即達 5% 顯著者的格數 | 每格各跑一次 bootstrap |

::: {.callout-important}

### 兩個最容易誤讀的欄位

**「pp」不是「%」。** 本研究的策略絕對年化報酬多落在 −0.6% 至 +0.9%，
而交易層的效果量中位為 +0.456 **pp**——**效果量與報酬水準本身同量級**。
這不矛盾：Δ 是兩臂相減，共同的市場成分已被消去（§3.5）。

**「逐格顯著 2/15」不是「只有 2 格有效」。** 15 格方向多數一致卻只有少數
單獨顯著，代表的是**單格樣本不足以偵測**，而非效果只出現在那幾格。
主張一律以等權組合為準（§3.5 報告口徑）。

:::

::: {.aside}
另有 $SE$（標準誤）與 MDE（最小可偵測效果）兩個量出現於 §4.1.3 與 §4.5，
其定義與推導於該處給出。
:::


# 4.1 分組層：限制強度與配對品質

## 主檢定：9 組對照，方向全部偏向 GICS

固定排序準則與交易端，唯一變因為分組方法。$\Delta = r_{ML} - r_{GICS}$，正值代表資料驅動分群較優。

| 分群 | 排序 | 年化Δ(pp) | 95% CI (pp) | $p$ | BH 校正 $p$ |
| :--- | :--- | ---: | :---: | ---: | ---: |
| Agglomerative | DTW | −0.229 | [−1.11, +0.65] | 0.607 | 0.607 |
| HDBSCAN | DTW | −0.387 | [−1.17, +0.40] | 0.335 | 0.377 |
| Agglomerative | SSD | −0.447 | [−1.38, +0.38] | 0.322 | 0.377 |
| Agglomerative | SDP | −0.451 | [−1.31, +0.35] | 0.281 | 0.377 |
| HDBSCAN | SSD | −0.715 | [−1.53, +0.01] | 0.068 | 0.179 |
| HDBSCAN | SDP | −0.721 | [−1.51, **−0.01**] | 0.061 | 0.179 |
| K-means | DTW | −0.900 | [−2.06, +0.21] | 0.118 | 0.213 |
| K-means | SSD | −0.913 | [−2.01, +0.06] | 0.080 | 0.179 |
| K-means | SDP | −1.000 | [−2.13, +0.01] | 0.065 | 0.179 |

**方向 9/9 偏向 GICS，BH 校正後無一顯著**（校正後最小 $p$ = 0.179）。

> 九組方向全部一致，此型態比任何單組的 $p$ 值更有訊息量。
> 惟九組彼此不獨立（每三組共用同一 GICS 臂），故不施加正式的方向一致性檢定。

:::  {.aside}
全程並行計算 Newey-West HAC 作為參數法對照，**17 組對照兩法結論完全一致**（校正後皆不顯著）；完整雙欄見 `results/analysis/` 各 CSV。
:::

## 兩種「不顯著」的性質不同

::: {.callout-important}

雙尾檢定不顯著 **不等於** 兩者相當。區分兩者只需看**區間寬度**。

:::

| 層 | $SE$ 中位 | 95% CI 寬中位 | 最小可偵測效果 | 對 0.3pp 的檢定力 |
| :--- | ---: | ---: | ---: | ---: |
| 分組層 | 0.449 | 1.762 pp | **1.259 pp** | **10%** |
| 交易層 | 0.223 | 0.874 pp | **0.624 pp** | 27% |

**分組層的 MDE（1.26 pp）大於 GICS 參照臂自身的等權年化（+0.419%）**
→ 連「效果大到足以翻轉參照臂損益方向」的差異都未必看得見。

> 分組層的 null **幾乎不帶資訊**；但其點估計方向 9/9 一致、量級中位 −0.715 pp，
> 已達 MDE 的一半以上——資料無法排除「資料驅動分群顯著較差」，
> 其程度遠甚於無法排除「兩者相當」。

## 分組限制強度：一條有序的維度

各分組方式對候選池施加的限制強度不同。依排除標的比例排列，並併入不分組零點：

| 分組方式 | 排除標的 | 強平率 | 平均 Sharpe | 等權年化 |
| :--- | ---: | ---: | ---: | ---: |
| **不分組** | **0%** | **36.8%** | **+0.081** | **+0.683%** |
| GICS 產業 | 15.1% | 39.6% | +0.024 | +0.419% |
| HDBSCAN | 26.4% | 40.5% | −0.116 | −0.186% |
| Agglomerative | 35.3% | 41.5% | −0.065 | +0.049% |
| K-means | 42.0% | 43.1% | −0.249 | −0.531% |

**逐格觀之**（平均 Sharpe）：

| 分組 | SSD | DTW | SSD-DTW-PCA |
| :--- | ---: | ---: | ---: |
| **不分組** | **+0.099** | **+0.083** | **+0.061** |
| GICS 產業 | +0.022 | +0.014 | +0.037 |
| Agglomerative | −0.092 | −0.056 | −0.048 |
| HDBSCAN | −0.153 | −0.066 | −0.128 |
| K-means | −0.271 | −0.241 | −0.234 |

::: {.callout-important}
**不分組三格全部優於同排序的 GICS；GICS 三格又全部優於任一資料驅動分群格。**
:::

> ⚠️ 非嚴格單調：Agglomerative 排除標的多於 HDBSCAN 卻表現較好——已知反例，
> 故稱之為趨勢而非定律。

# 4.2 交易層：門檻選擇 vs. 固定門檻

## 主檢定：五種配對底方向全正，校正後 0/5 顯著

對照的兩臂為 **DL-THR**（逐期由模型選擇進場門檻）與 **Z-Score**（固定 $z$=2.0）。
兩臂共用同一批配對、同一組參數格，唯一變因是門檻如何決定。

| 配對底 | 年化Δ(pp) | 95% CI (pp) | IR | 勝日% | $p$ | BH 校正 $p$ |
| :--- | ---: | :---: | ---: | ---: | ---: | ---: |
| **GICS × SSD（傳統）** | **+0.573** | [+0.09, +1.19] | 0.427 | 51.5 | 0.042 | 0.087 |
| **GICS × SDP（傳統）** | **+0.495** | [+0.08, +0.96] | 0.357 | 51.7 | 0.028 | 0.087 |
| HDBSCAN × SDP | +0.456 | [+0.04, +0.96] | 0.344 | 51.2 | 0.052 | 0.087 |
| K-means × SSD | +0.283 | [−0.04, +0.58] | 0.223 | 52.5 | 0.073 | 0.091 |
| Agglomerative × SSD | +0.112 | [−0.20, +0.44] | 0.086 | 51.5 | 0.488 | 0.488 |

::: {.callout-important}

**方向 5/5 為正，但 BH 校正後 0/5 顯著**（校正前 2/5）。

觀測效果中位 +0.456 pp 對最小可偵測效果 0.62 pp
→ 屬「效果存在但接近偵測極限」，非「檢定力不足到無從判斷」。

:::

> **增益最大者為兩條傳統 GICS 配對底** → 門檻選擇的改良與「配對如何找到」**正交**，
> 不依賴資料驅動分群，在傳統產業分組上反而取得最大增益。

::: {.aside}
五輪獨立重訓的跨輪全距為 0.010–0.047 個 Sharpe 單位，遠小於增益本身——訓練隨機性不是結果的來源。
:::

## 單一參數配置的檢定力限制

等權組合的解析度來自平均掉各配置的特異噪音。逐格檢定：

| 配對底 | 15 格中正向顯著 | 年化Δ 中位 (pp) | 年化Δ 全距 (pp) |
| :--- | :---: | ---: | :---: |
| HDBSCAN × SDP | 5/15 | +0.623 | −0.37 ~ +0.96 |
| GICS × SDP | 5/15 | +0.506 | −0.56 ~ +1.61 |
| K-means × SSD | 3/15 | +0.335 | −0.29 ~ +0.75 |
| Agglomerative × SSD | 2/15 | +0.040 | −0.50 ~ +0.74 |
| GICS × SSD | 2/15 | +0.520 | −0.76 ~ +1.84 |

**單一參數配置的檢定力顯著低於等權組合**——全距普遍橫跨零。

> 此即 3.5.5 節採等權口徑的理由之一：
> 逐格報告不僅檢定力低，且「挑最好的那一格」會引入選擇偏誤。

## 增益來源①：不是拉高門檻

DL-THR 實際進場門檻中位數 **2.11–2.33**，高於基準 2.0。
若增益僅來自「調高門檻」，把 Z-Score 拉到同一水準應能複製。

| 配對底 | A：DL−ZS(2.0) | B：DL−ZS(同門檻) | C：門檻管道 | **複製率** |
| :--- | ---: | ---: | ---: | ---: |
| GICS-SSD | +0.573 | +0.502 ~ +0.575 | −0.002 ~ +0.071 | **−0.3 ~ 12.4%** |
| GICS-SDP | +0.495 | +0.407 ~ +0.445 | +0.050 ~ +0.088 | 10.1 ~ 17.8% |
| K-means | +0.283 | +0.198 ~ +0.246 | +0.037 ~ +0.085 | 13.1 ~ 30.0% |

**門檻管道僅能複製表面增益的 0–30%**，且對照 B 的效果量幾乎未縮水
（GICS-SSD 自 +0.573 僅降至 +0.502）。

> 增益不是「單純調高門檻」的效果。

## 增益來源②：不是 SKIP 的選股方向

DL-THR 的 SKIP 率為 **32.9–38.1%**。

::: {.callout-important}

底層策略期望值為負 → **隨機跳過任一批配對，期望上都會「避開損失」**
→ 必須以置換檢定建立虛無分布（每格 2,000 次不放回重抽）。

:::

| 配對底 | 實際避損 | 隨機期望 | 技巧成分 | 平均百分位 | 顯著格數 |
| :--- | ---: | ---: | ---: | ---: | :---: |
| Agglomerative | −12.9 | +139.2 | **−152.0** | 36.7 | 0/15 |
| K-means | +338.9 | +461.2 | **−122.3** | 38.6 | 0/15 |
| GICS-SDP | +78.3 | −237.6 | +315.9 | 63.9 | 1/15 |
| GICS-SSD | +115.1 | −199.8 | +314.8 | 66.6 | 1/15 |
| HDBSCAN | +409.9 | +235.1 | +174.8 | 65.3 | 2/15 |

**75 格中僅 4 格顯著（隨機期望 3.75）** → SKIP 的選股方向不具可證實的技巧。
Agglomerative 與 K-means 的技巧成分甚至為**負**。

## 增益來源③：也不是總曝險減少

前一項對齊的是**門檻**而非**曝險**：即使門檻拉到 2.3，
Z-Score 的進場次數仍為 3,115–3,340，而 DL-THR 為 2,107–2,464。

故另建虛無分布——**固定門檻 2.0，但隨機跳過與 DL-THR 同數量的配對期**：

| 配對底 | ZS 全額 | 隨機跳過 | DL-THR 實際 | 超額 | 百分位 | 顯著格數 |
| :--- | ---: | ---: | ---: | ---: | ---: | :---: |
| **GICS-SSD** | +904 | +705 | **+2,334** | **+1,629** | 88.3 | **11/15** |
| **GICS-SDP** | +1,281 | +1,036 | **+2,515** | **+1,480** | 85.1 | **10/15** |
| HDBSCAN | −518 | −283 | +651 | +934 | 80.2 | 8/15 |
| K-means | −1,374 | −912 | −668 | +244 | 66.0 | 4/15 |
| Agglomerative | −210 | −64 | −72 | −8 | 51.0 | 4/15 |

> 兩條 GICS 底的 **ZS 全額高於隨機跳過**（+904 > +705）——
> 對期望值為正的底層策略，少交易本身是虧的，
> 故「降低曝險」不可能是其增益來源。

::: {.aside}
三項檢定共同侷限：皆為事後重抽，不含槽位再配置效應。
:::

## 受控對照：反事實標籤值多少錢？

DL-THR 的一項結構性優勢是**全資訊**——9 個動作的報酬皆可精確反事實回算。
RL-THR 保持動作選單、狀態、網路與 walk-forward 切分完全相同，
只把訓練標籤縮成「實際選中的那一個」並改採 $\varepsilon$-greedy（設計見 §3.4）。

配對底 `Grid (AGG-SSD)`，15 格等權：

| 交易端 | 平均 Sharpe | 等權年化 | vs Z-Score |
| :--- | ---: | ---: | ---: |
| Z-Score（固定門檻） | −0.092 | −0.112% | — |
| **DL-THR**（全資訊） | −0.015 | −0.055% | +0.057 pp |
| **RL-THR** ($\varepsilon$=0.05) | **+0.076** | **+0.215%** | **+0.326 pp** |
| RL-THR ($\varepsilon$=0.10) | +0.073 | +0.185% | +0.297 pp |
| RL-THR ($\varepsilon$=0.20→0.02) | +0.063 | +0.162% | +0.274 pp |

::: {.callout-warning}
**三個 RL-THR 變體全部優於 DL-THR。**

反事實標籤的價值在此配對底上**為負**——全資訊不僅沒有幫助，
還可能使模型過度擬合到那些從未被選中的動作之報酬估計上。
:::

> 此結果限於 Agglomerative 配對底（唯一設有 RL-THR 對照者），
> 且未施加統計檢定。但方向足以推翻「全資訊是 DL-THR 有效之來源」的推測——
> 該推測原是本研究命名 DL-THR 而非 DRL 的理由之一。

# 4.3 組合系統：實務上要部署的那個檢定

4.1 與 4.2 各測一個成分，**都不是實務上要部署的系統**。

> **組合系統**　資料驅動分群 + 排序 + 篩選 + **DL-THR 交易端**
>
> **傳統基準**　GICS 產業分組 + 同一排序 + 同一篩選 + **固定門檻 Z-Score**

**全期（2001–2025）**

| 分群法 | 傳統基準 | 年化Δ(pp) | IR | 95% CI (pp) | $p$ | BH 校正 $p$ |
| :--- | :--- | ---: | ---: | :---: | ---: | ---: |
| Agglomerative | GICS-SSD | **−0.334** | −0.128 | [−1.28, +0.53] | 0.469 | 0.558 |
| HDBSCAN | GICS-SDP | **−0.265** | −0.105 | [−1.20, +0.59] | 0.558 | 0.558 |
| K-means | GICS-SSD | **−0.630** | −0.227 | [−1.73, +0.36] | 0.228 | 0.558 |

::: {.callout-important}
**三組方向全部為負，校正後無一顯著。**
完整系統**劣於**傳統基準——原因見下頁的成分分解。
:::

## 組合系統（續）：2012 年後

排除 2008 金融海嘯與其後的高波動期後重跑同一組對照：

| 分群法 | 傳統基準 | 年化Δ(pp) | IR | 95% CI (pp) | $p$ | BH 校正 $p$ |
| :--- | :--- | ---: | ---: | :---: | ---: | ---: |
| Agglomerative | GICS-SSD | +0.470 | 0.259 | [−0.36, +1.21] | 0.234 | 0.350 |
| HDBSCAN | GICS-SDP | +0.193 | 0.110 | [−0.53, +0.84] | 0.587 | 0.587 |
| K-means | GICS-SSD | +0.582 | 0.306 | [−0.21, +1.31] | 0.135 | 0.350 |

**方向翻正，但仍無一顯著。**

> 全期為負、2012 後為正，差異來自 2008–2011 這段。
> 該期間的分組層損害最大——與 4.6.1 的 regime 分層一致：
> 動盪期是策略唯一獲利的環境，而分組限制在該環境的代價也最高。

## 成分分解：損害來自哪一半？

$$(\text{分群}+\text{DL})-(\text{GICS}+\text{ZS}) = \underbrace{(\text{分群}+\text{DL})-(\text{分群}+\text{ZS})}_{\text{DL-THR 成分}} + \underbrace{(\text{分群}+\text{ZS})-(\text{GICS}+\text{ZS})}_{\text{分群成分}}$$

| 期間 | 分群法 | 總效果 | DL-THR 成分 | $p$ | 分群成分 | $p$ |
| :--- | :--- | ---: | ---: | ---: | ---: | ---: |
| 全期 | Agglomerative | −0.334 | **+0.112** | 0.488 | **−0.447** | 0.322 |
| 全期 | HDBSCAN | −0.265 | **+0.456** | 0.052 | **−0.721** | 0.061 |
| 全期 | K-means | −0.630 | **+0.283** | 0.073 | **−0.913** | 0.080 |
| 2012+ | Agglomerative | +0.470 | +0.279 | 0.064 | +0.191 | 0.590 |
| 2012+ | HDBSCAN | +0.193 | +0.511 | 0.013 | −0.318 | 0.387 |
| 2012+ | K-means | +0.582 | +0.424 | 0.009 | +0.158 | 0.687 |

::: {.callout-important}
**全期三組的 DL-THR 成分皆正、分群成分皆負且量級更大。**

系統劣於基準的原因是**分群層的損害超過交易端的貢獻**——兩成分並不互補。
:::

（分解為恆等式，最大殘差 0.001 pp。）

# 4.4 分組層為何是淨損害：期末強制平倉機制

## 損益結構：兩股對衝流量的殘差

489 個 Z-Score 配置的逐筆交易統計（計數器修正後，見附錄 B.2）：

| 指標 | 中位數 |
| :--- | ---: |
| 逐筆勝率 | **0.578** |
| 獲利因子 | 0.969 |
| 期末強制平倉 ÷ 進場 | **0.418** |

**多數交易確實收斂獲利，但獲利被少數大額虧損吃光。**

以 GICS-SSD 為例（15 配置平均）：收斂獲利 **+15,199**、強平虧損 **−8,424**、
淨額僅 **+6,775**。淨額佔收斂獲利的比例隨限制強度單調下降：
不分組 48.2% → K-means 36.8%。

> 典型個案（Top 1，第一期 UNH／CI，$\beta$=0.9638）：
> $z$=2.06 進場 → 價差擴大至 $z$=5.46 → 95 日後期末強平，單筆虧 10.9%。

## 強平率與績效：15 個配置一致為負

期末強平率隨分組限制強度**單調上升**，且該序**無**績效序的反例：

| 分組方式 | 排除標的 | 強平率 | 平均 Sharpe |
| :--- | ---: | ---: | ---: |
| **不分組** | **0%** | **36.8%** | **+0.081** |
| GICS 產業 | 15.1% | 39.6% | +0.024 |
| HDBSCAN | 26.4% | 40.5% | −0.116 |
| Agglomerative | 35.3% | 41.5% | −0.065 |
| K-means | 42.0% | 43.1% | −0.249 |

| 層級 | 相關 |
| :--- | :--- |
| 逐臂（15 臂） | Pearson $r$ = **−0.895**（$p$<0.0001） |
| 逐配置內、跨 15 臂 | **15/15 為負**，中位 $r$ = −0.801 |

::: {.callout-warning}
**全樣本混合（n=225）的相關為 +0.201，符號相反**——停損維度造成的 Simpson 悖論
（SL5% 強平率 32.8%／Sharpe −0.286；SL0% 45.2%／+0.115）。**不可引用混合值。**
:::

## 強平的配對只是「尚未回歸」嗎？

對 **4,738 筆**期末強制平倉的交易，往後追蹤 126 個交易日（一個完整交易期）：

| 追蹤期 | 累計回歸比例 |
| :--- | ---: |
| 21 日 | 13.8% |
| 42 日 | 22.2% |
| 63 日 | 28.1% |
| **126 日** | **39.1%** |

平倉時 $|z|$ 中位 **3.82**。再給一個完整交易期，**僅 39.1% 回歸**；
未回歸的 **60.9%**，其 $|z|$ 自 4.88 **繼續擴大至 6.61**。

::: {.callout-important}
**這些配對不是尚未回歸，是持續發散。**

延長交易期會讓四成轉盈、六成擴大虧損 → 已否證（附錄 C.3）。
:::

> **機制鏈**：限制候選池 → 選中的配對出樣本收斂性差 → 期末強平實現大額虧損
> → 吃掉收斂交易的獲利。**分組有害不是因為分錯，而是因為縮小了選擇空間。**

# 4.5 兩層改良的檢定力不對等

以主檢定的信賴區間反推標準誤（$SE = $ 區間寬 $/\,3.92$）：

| 層 | 對照組數 | $\lvert\Delta\rvert$ 中位 | $SE$ 中位 | **MDE** |
| :--- | ---: | ---: | ---: | ---: |
| 形成期（分組層） | 9 | 0.715 pp | 0.449 | **1.259 pp** |
| 交易期（交易層） | 5 | 0.456 pp | 0.223 | **0.624 pp** |

**形成期層的 $SE$ 是交易期層的 2.02 倍** → 等效需 **4.1 倍**樣本期間。

**成因是設計，不是資料量：**

| 層 | 兩臂的配對 | 差分消去了什麼 |
| :--- | :--- | :--- |
| 交易期 | **共用同一批** | 市場衝擊 + 配對特異變異 |
| 形成期 | **不同集合** | 僅市場衝擊；標的特異變異殘留 |

> 欲把 MDE 壓到 0.3 pp：形成期層需 **17.6 倍**樣本（逾四個世紀的日資料），
> 交易期層需 4.3 倍。**更換演算法、增加特徵、改良插補皆不影響此限制。**
> 唯一出路是構造使兩臂共用同一批標的的配對設計。

## 篩選層與分組層的交互作用

文獻多將「分群 + 共整合篩選」並用，未討論兩者的交互作用。排序固定 SSD：

| 分組 | 統計篩選 | 產業 one-hot | 平均 Sharpe | 等權年化 |
| :--- | :---: | :---: | ---: | ---: |
| 不分組 | ADF | — | **+0.099** | **+0.715%** |
| 不分組 | 無 | — | +0.028 | +0.190% |
| Agglomerative | ADF | 1.0 | −0.092 | −0.112% |
| Agglomerative | 無 | 1.0 | −0.078 | −0.296% |
| Agglomerative | ADF | 0 | −0.178 | −0.107% |
| Agglomerative | 無 | 0 | −0.009 | **+0.306%** |

::: {.callout-important}
**篩選層的價值取決於候選池規模。**

不分組時施加篩選：+0.190% → **+0.715%**（有益）
分群且無產業先驗時施加篩選：+0.306% → **−0.107%**（有害）
:::

> 四項條件全開（分群 + 篩選 + 產業先驗，即多數文獻的預設）為 −0.112%，
> 比表中任一格都差。

::: {.aside}
本頁為描述性比較，未施加統計檢定；最大差距 0.82 pp 低於 4.5 節量化的 1.26 pp 偵測門檻。
:::

# 4.6 風險評估

## Regime 分層

口徑同全章：**15 格等權組合**，且**逐格對齊兩臂**
→ Z-Score 與 DL-THR 之間的唯一變因仍是交易端。

| 配對底 | 交易端 | Calm | Normal | Turbulent |
| :--- | :--- | ---: | ---: | ---: |
| GICS-SSD | Z-Score | −0.41 | −0.50 | **+0.77** |
| GICS-SSD | DL-THR | −0.23 | −0.26 | **+0.75** |
| GICS-SDP | Z-Score | −0.25 | −0.47 | **+0.76** |
| GICS-SDP | DL-THR | −0.05 | −0.15 | **+0.68** |
| Agglomerative | Z-Score | −0.47 | −0.66 | +0.55 |
| Agglomerative | DL-THR | −0.23 | −0.43 | +0.31 |
| K-means | Z-Score | −0.45 | −0.90 | +0.26 |
| K-means | DL-THR | −0.06 | −0.55 | +0.11 |

**十列型態一致：動盪期為正，平靜期與一般期為負。**
DL-THR 25 格中改善 19 格，集中於虧損較大的兩檔。

::: {.callout-warning}
**此型態不支持「僅於高波動期交易」的策略**——動盪期報酬的 54–87%
集中於 2008–2009 兩年；2022 年動盪日數最多卻近乎零報酬（附錄 C.1）。
:::

## 交易成本敏感度：餘裕極薄

成本模型可解析求解：進出場費用 = friction × 名目額，且名目額恰等於每配對資金。
口徑同上頁：15 格等權、逐格對齊兩臂。

| 配對底 | Z-Score 往返 BE% | DL-THR 往返 BE% | Z 餘裕 | DL 餘裕 |
| :--- | ---: | ---: | ---: | ---: |
| GICS-SDP | 0.612 | 0.669 | +3.2 bps | **+8.9 bps** |
| GICS-SSD | 0.603 | 0.667 | +2.3 bps | **+8.7 bps** |
| HDBSCAN | 0.567 | 0.605 | −1.3 bps | +2.5 bps |
| Agglomerative | 0.574 | 0.577 | −0.6 bps | −0.3 bps |
| K-means | 0.541 | 0.552 | −3.9 bps | −2.8 bps |

現行成本假設為往返 **0.58%**（單邊 29 bps）。

::: {.callout-important}
**五個配對底的 DL-THR 皆提高 break-even**（0.3–6.3 bps），
但餘裕全距僅 **−3.9 ~ +8.9 bps**——三個資料驅動分群底的 Z-Score 臂已為負。

相對於 58 bps 的成本水準，8.9 bps 的餘裕仍屬極薄。
:::

## 絕對績效：全部低於無風險利率

依 3.5.5 節口徑，以 15 個參數配置的等權組合報告。主軸 15 格等權年化：

| 分組 | SSD | DTW | SSD-DTW-PCA |
| :--- | ---: | ---: | ---: |
| **不分組** | +0.715% | **+0.812%** | +0.521% |
| GICS 產業 | +0.346% | +0.415% | +0.495% |
| Agglomerative | −0.112% | +0.199% | +0.059% |
| HDBSCAN | −0.391% | +0.049% | −0.215% |
| K-means | −0.583% | −0.499% | −0.511% |

疊加 DL-THR 後最佳者為 **GICS-SDP**：年化 +0.914%、25 年終值 **12,548**、
動用資本口徑 +2.161%、平均 Sharpe 0.217。

::: {.callout-important}
**同期僅持有無風險資產（2% 假設）將得約 16,400。**

本研究的任何配置皆未達此水準。
:::

**Deflated Sharpe**（$N$=44、$SR_0$=0.328）：最高為不分組×DTW 的 **0.769**，
**無一通過 0.95**。

## 相對比較與絕對績效為何是兩件事

| | 逐日差分 bootstrap | 絕對 bootstrap |
| :--- | :--- | :--- |
| $H_0$ | 兩臂績效相同 | 策略平均日報酬為零 |
| 對照物 | 同配對、同參數格的另一臂 | **零** |
| 主張性質 | **相對** | **絕對** |

配對設計消去共同的市場風險 → 訊噪比大幅提高；
絕對檢定沒有對照物可消噪，其標準誤必然大得多。

::: {.callout-important}
本章的三項檢定**全部是相對比較**。即使某一項達顯著，
也只代表「A 優於 B」，不代表 A 本身可獲利。

而本研究的絕對績效**明確為負**（低於無風險利率），
故連「相對較優者是否值得部署」都不成立。
:::

> 這使本研究的定位清楚：**方法論研究**，
> 回答「哪一層的改良可被驗證、哪一層不能」，
> 而非提出一個可交易的策略。

## 本章小結

| 檢定 | 內容 | 方向 | 校正後顯著 |
| :--- | :--- | :--- | :---: |
| 分組層 | 資料驅動分群 vs GICS | 9/9 偏向 GICS | **0/9** |
| 交易層 | DL-THR vs 固定門檻 | 5/5 為正 | **0/5** |
| 兩層組合 | 完整系統 vs 傳統基準 | 全期 3/3 為負 | **0/3** |

**方向的一致性是最可靠的訊號，而非任何單組的 $p$ 值。**

::: {.callout-important}

**其一，分組層是淨損害**，且限制愈強損害愈大——
不分組 > GICS > 任一資料驅動分群，15 格逐格成立。
機制為期末強制平倉：強平率自 36.8% 升至 43.1%，
與績效的相關在 15 個參數配置中一致為負（中位 $r$ = −0.801）。
強平的配對再追一個交易期僅 39.1% 回歸——**是持續發散，非尚未回歸**。

**其二，門檻選擇是唯一方向為正的改良**，且價值與分群無關：
增益最大者為傳統 GICS 底，三項機械性替代解釋在該兩底皆排除。
惟量級接近偵測極限，校正後不顯著。

**其三，兩層的檢定力不對等來自設計**：
形成期層的 $SE$ 為交易期層的 2.02 倍，等效需 4.1 倍樣本。
此為結構性限制，更換演算法或增加特徵皆不影響。

:::

::: {.callout-warning}
**絕對績效**：最佳配置（GICS-SDP + DL-THR）25 年將 10,000 變為 **12,548**，
低於同期無風險利率假設的約 **16,400**；DSR 無一通過 0.95。

**本研究不主張任何配置具可交易的獲利能力；全部結論均為相對宣稱。**
:::